# Vídeo con YOLO: conteo de objetos por clase
Este notebook se centra en una tarea muy habitual en visión aplicada: detectar objetos en un vídeo y contar cuántos aparecen de cada clase. Es una continuación natural tras la detección básica con YOLO.

## Objetivo
Queremos responder preguntas como estas:
- ¿Cuántas personas aparecen en una escena?
- ¿Cuántos coches detecta el modelo en cada frame?
- ¿Qué clases aparecen con más frecuencia a lo largo del vídeo?

En este notebook haremos un conteo básico por frame y un resumen global.

## Requisitos
- `ultralytics`
- `opencv-python`
- `matplotlib`
- entorno local si quieres usar `cv2.imshow`

In [ ]:
# Instalación orientativa si hace falta
# !pip install -U ultralytics opencv-python

In [ ]:
from collections import Counter, defaultdict
from pathlib import Path
import os

import cv2
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Video, display, Markdown

try:
    from ultralytics import YOLO
except Exception as e:
    print("Ultralytics no disponible:", e)

print("Directorio actual:", os.getcwd())

## Selección del vídeo

In [ ]:
VIDEO_PATH = Path("video.mp4")

try:
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        VIDEO_PATH = Path(next(iter(uploaded)))
except Exception:
    pass

print(f"Vídeo seleccionado: {VIDEO_PATH}")
print(f"¿Existe?: {VIDEO_PATH.exists()}")

In [ ]:
if VIDEO_PATH.exists():
    display(Video(str(VIDEO_PATH), embed=True))

## Carga del detector

In [ ]:
model = YOLO("yolov8n.pt")
class_names = model.names
class_names

## Conteo por frame
Cada frame se procesa con YOLO y se cuentan las detecciones por clase. Después se agregan esos conteos para sacar resúmenes.

In [ ]:
frame_counts = []
max_counts = defaultdict(int)
total_detections = Counter()

cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():
    raise FileNotFoundError(f"No se pudo abrir el vídeo: {VIDEO_PATH}")

frame_idx = 0
sampled_results = []

while True:
    ret, frame = cap.read()
    if not ret or frame is None:
        break

    results = model.predict(source=frame, verbose=False)[0]
    counts = Counter()

    if results.boxes is not None and len(results.boxes) > 0:
        cls_ids = results.boxes.cls.cpu().numpy().astype(int)
        for cls_id in cls_ids:
            label = class_names[int(cls_id)]
            counts[label] += 1
            total_detections[label] += 1
            if counts[label] > max_counts[label]:
                max_counts[label] = counts[label]

    frame_counts.append(counts)

    if frame_idx % 30 == 0:
        sampled_results.append((frame_idx, counts.copy(), results.plot()))

    frame_idx += 1

cap.release()
print(f"Frames procesados: {frame_idx}")
print(f"Clases detectadas: {list(total_detections.keys())}")

## Resumen global

In [ ]:
print("Total de detecciones acumuladas por clase:")
for label, value in total_detections.most_common():
    print(f"- {label}: {value}")

print("\nMáximo número detectado en un mismo frame:")
for label, value in sorted(max_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"- {label}: {value}")

In [ ]:
if total_detections:
    labels = [k for k, _ in total_detections.most_common()]
    values = [total_detections[k] for k in labels]

    plt.figure(figsize=(10, 4))
    plt.bar(labels, values)
    plt.xticks(rotation=45, ha="right")
    plt.title("Detecciones acumuladas por clase")
    plt.ylabel("Número de detecciones")
    plt.tight_layout()
    plt.show()

## Visualización de algunos frames anotados

In [ ]:
if sampled_results:
    cols = min(3, len(sampled_results))
    rows = int(np.ceil(len(sampled_results) / cols))
    plt.figure(figsize=(15, 4 * rows))

    for i, (frame_idx, counts, annotated) in enumerate(sampled_results, start=1):
        plt.subplot(rows, cols, i)
        plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        title = ", ".join([f"{k}:{v}" for k, v in counts.items()]) if counts else "sin detecciones"
        plt.title(f"Frame {frame_idx} | {title}")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

## Conteo sobre vídeo en tiempo real
Esta variante dibuja en cada frame el número de objetos detectados por clase.

In [ ]:
# Conteo por frame con visualización local
cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():
    raise FileNotFoundError(f"No se pudo abrir el vídeo: {VIDEO_PATH}")

while True:
    ret, frame = cap.read()
    if not ret or frame is None:
        break

    results = model.predict(source=frame, verbose=False)[0]
    annotated = results.plot()

    counts = Counter()
    if results.boxes is not None and len(results.boxes) > 0:
        cls_ids = results.boxes.cls.cpu().numpy().astype(int)
        for cls_id in cls_ids:
            counts[class_names[int(cls_id)]] += 1

    y = 25
    for label, value in counts.most_common(5):
        cv2.putText(annotated, f"{label}: {value}", (10, y), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
        y += 25

    cv2.imshow("Conteo YOLO", annotated)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

## Qué significa realmente "contar"
En este notebook contamos detecciones. Eso no siempre equivale a contar objetos únicos, porque el mismo coche o la misma persona puede aparecer en muchos frames. Para conteo de objetos únicos, normalmente hay que combinar detección con tracking e incluso con líneas o zonas de paso.

In [ ]:
Markdown('''
**Actividad sugerida**

1. Ejecuta el detector sobre un vídeo de tráfico o de personas caminando.
2. Observa qué clases detecta mejor y cuáles confunde o no encuentra.
3. Compara el total de detecciones con el máximo número observado por frame.
4. Como ampliación, añade tracking para intentar contar objetos únicos.
''')